In [25]:
#Cài đặt thuật giải AKT vào bài toán 15 puzzle(n=4)
import copy
from heapq import heappush, heappop

n = 4
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]
goal = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 0]]

def find_zero(state):
    for i in range(n):
        for j in range(n):
            if state[i][j] == 0:
                return i, j
    return None

def heuristic(state):
    h = 0
    for i in range(n):
        for j in range(n):
            val = state[i][j]
            if val != 0:
                goal_x = (val - 1) // n
                goal_y = (val - 1) % n
                h += abs(i - goal_x) + abs(j - goal_y)
    return h

class Node:
    def __init__(self, state, parent, g, h):
        self.state = state
        self.parent = parent
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f

class PriorityQueue:
    def __init__(self):
        self.queue = []
    def push(self, node):
        heappush(self.queue, node)
    def pop(self):
        return heappop(self.queue)
    def is_empty(self):
        return len(self.queue) == 0

def is_safe(x, y):
    return 0 <= x < n and 0 <= y < n

def print_path(node):
    if node is None:
        return
    print_path(node.parent)
    for row in node.state:
        print(row)
    print("------")

def AKT(start):
    open_list = PriorityQueue()
    closed = set()
    root = Node(start, None, 0, heuristic(start))
    open_list.push(root)

    while not open_list.is_empty():
        current = open_list.pop()
        state_tuple = tuple(tuple(row) for row in current.state)

        if state_tuple in closed:
            continue

        closed.add(state_tuple)

        if current.state == goal:
            print("Tìm thấy lời giải!")
            print_path(current)
            return

        x, y = find_zero(current.state)
        for i in range(4):
            new_x, new_y = x + rows[i], y + cols[i]
            if is_safe(new_x, new_y):
                temp_state = copy.deepcopy(current.state)
                temp_state[x][y], temp_state[new_x][new_y] = temp_state[new_x][new_y], temp_state[x][y]
                new_tuple = tuple(tuple(row) for row in temp_state)
                if new_tuple not in closed:
                    child = Node(temp_state, current, current.g + 1, heuristic(temp_state))
                    open_list.push(child)
    print("Không có lời giải.")

In [26]:
start_board = [
    [1, 2, 3, 4],
    [5, 6, 0, 8],
    [9, 10, 7, 11],
    [13, 14, 15, 12]
]

print("--- TRẠNG THÁI BẮT ĐẦU ---")
for r in start_board: print(r)

print("\n--- ĐANG GIẢI BẰNG AKT ---")
AKT(start_board)

--- TRẠNG THÁI BẮT ĐẦU ---
[1, 2, 3, 4]
[5, 6, 0, 8]
[9, 10, 7, 11]
[13, 14, 15, 12]

--- ĐANG GIẢI BẰNG AKT ---
Tìm thấy lời giải!
[1, 2, 3, 4]
[5, 6, 0, 8]
[9, 10, 7, 11]
[13, 14, 15, 12]
------
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 0, 11]
[13, 14, 15, 12]
------
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 0]
[13, 14, 15, 12]
------
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]
------


In [28]:
#Cài đặt thuật giải AKT vào bài toán 15 puzzle(n>4).
import copy
from heapq import heappush, heappop

def find_zero(state, n):
    for i in range(n):
        for j in range(n):
            if state[i][j] == 0:
                return i, j
    return None

def heuristic(state, n, goal_map):
    h = 0
    for i in range(n):
        for j in range(n):
            val = state[i][j]
            if val != 0:
                goal_x, goal_y = goal_map[val]
                h += abs(i - goal_x) + abs(j - goal_y)
    return h

class Node:
    def __init__(self, state, parent, g, h):
        self.state = state
        self.parent = parent
        self.g = g
        self.h = h
        self.f = g + h
    def __lt__(self, other):
        return self.f < other.f

def solve_n_puzzle(start_state, n):
    # Tạo trạng thái đích động cho nxn
    goal = []
    goal_map = {}
    count = 1
    for i in range(n):
        row = []
        for j in range(n):
            if count < n*n:
                row.append(count)
                goal_map[count] = (i, j)
                count += 1
            else:
                row.append(0)
        goal.append(row)

    rows = [1, 0, -1, 0]
    cols = [0, -1, 0, 1]

    open_list = []
    heappush(open_list, Node(start_state, None, 0, heuristic(start_state, n, goal_map)))
    closed = set()

    while open_list:
        current = heappop(open_list)
        state_tuple = tuple(tuple(r) for r in current.state)

        if state_tuple in closed: continue
        closed.add(state_tuple)

        if current.state == goal:
            return current

        x, y = find_zero(current.state, n)
        for i in range(4):
            nx, ny = x + rows[i], y + cols[i]
            if 0 <= nx < n and 0 <= ny < n:
                new_state = [list(r) for r in current.state]
                new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]
                if tuple(tuple(r) for r in new_state) not in closed:
                    h_val = heuristic(new_state, n, goal_map)
                    heappush(open_list, Node(new_state, current, current.g + 1, h_val))
    return None

# Ví dụ n = 5 (24-puzzle)
n_size = 5
# Trạng thái bắt đầu (ví dụ đơn giản)
start = [[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20], [21, 22, 23, 0, 24]]

result = solve_n_puzzle(start, n_size)
if result:
    print(f"Thành công giải n={n_size} puzzle!")
    # In bớc cuối
    for r in result.state: print(r)
else:
    print("Không tìm thấy lời giải.")

Thành công giải n=5 puzzle!
[1, 2, 3, 4, 5]
[6, 7, 8, 9, 10]
[11, 12, 13, 14, 15]
[16, 17, 18, 19, 20]
[21, 22, 23, 24, 0]
